# Tutorial 1: Introduction to UTAC (Universal Threshold Activation)

**Feldtheorie V6 - Tutorial Series**

Welcome to the UTAC framework! This tutorial introduces the core concepts of Universal Threshold Activation and shows you how to fit β parameters to empirical data.

## Learning Objectives

By the end of this tutorial, you will:
1. Understand the UTAC sigmoid function σ(R; β, Θ)
2. Fit β and Θ parameters to real data
3. Interpret the physical meaning of β (steepness) and Θ (threshold)
4. Visualize threshold behavior across different systems

---

## 1. Setup and Imports

In [ ]:
# Standard imports
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# UTAC Framework
from models.sigmoid_fit import fit_sigmoid, sigmoid

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Imports successful")

## 2. The UTAC Sigmoid Function

The core of UTAC is the sigmoid response function:

$$
\sigma(R; \beta, \Theta) = \frac{1}{1 + e^{-\beta(R - \Theta)}}
$$

Where:
- **R**: Resource/activation parameter (control parameter)
- **β**: Steepness parameter (critical exponent)
- **Θ**: Threshold location (critical point)

Let's visualize this:

In [ ]:
# Create resource range
R = np.linspace(0, 20, 500)
Theta = 10.0

# Different β values
betas = [2.0, 4.2, 8.0]
colors = ['blue', 'orange', 'green']
labels = [r'$\beta = 2$', r'$\beta = 4.2$ ($\Phi^3$)', r'$\beta = 8$']

plt.figure(figsize=(10, 6))

for beta, color, label in zip(betas, colors, labels):
    sigma = sigmoid(R, beta, Theta)
    plt.plot(R, sigma, color=color, linewidth=2.5, label=label)

# Mark threshold
plt.axvline(Theta, color='red', linestyle='--', linewidth=1.5, label=r'$\Theta = 10$')
plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

plt.xlabel('Resource/Activation $R$', fontsize=12)
plt.ylabel('Response $\sigma(R)$', fontsize=12)
plt.title('UTAC Sigmoid: Effect of β on Transition Steepness', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Higher β → Sharper transition (more 'catastrophic')")
print("✓ Lower β → Gradual transition (more 'smooth')")

## 3. Fitting β to Empirical Data

Let's fit the UTAC model to synthetic data representing a climate tipping point.

In [ ]:
# Generate synthetic data (simulating AMOC collapse)
np.random.seed(42)
R_data = np.linspace(0, 20, 50)
beta_true = 4.0
theta_true = 10.0

# True signal + noise
sigma_true = sigmoid(R_data, beta_true, theta_true)
sigma_observed = sigma_true + np.random.normal(0, 0.05, len(R_data))
sigma_observed = np.clip(sigma_observed, 0, 1)  # Keep in [0,1]

# Fit the model
try:
    beta_fit, theta_fit, r_squared = fit_sigmoid(R_data, sigma_observed)
    print("✓ Fit successful!")
    print(f"  True β: {beta_true:.2f}, Fitted β: {beta_fit:.2f}")
    print(f"  True Θ: {theta_true:.2f}, Fitted Θ: {theta_fit:.2f}")
    print(f"  R² = {r_squared:.4f}")
except Exception as e:
    print(f"✗ Fit failed: {e}")
    beta_fit, theta_fit = beta_true, theta_true

# Plot results
plt.figure(figsize=(10, 6))
plt.scatter(R_data, sigma_observed, s=50, alpha=0.6, label='Observed Data', color='blue')
R_smooth = np.linspace(0, 20, 500)
plt.plot(R_smooth, sigmoid(R_smooth, beta_fit, theta_fit), 
         'r-', linewidth=2, label=f'Fitted: β={beta_fit:.2f}, Θ={theta_fit:.2f}')
plt.plot(R_smooth, sigma_true, 'g--', linewidth=1.5, alpha=0.7, label=f'True: β={beta_true:.2f}, Θ={theta_true:.2f}')
plt.axvline(theta_fit, color='red', linestyle=':', alpha=0.5)

plt.xlabel('Control Parameter $R$', fontsize=12)
plt.ylabel('Order Parameter $\sigma$', fontsize=12)
plt.title('UTAC Fit to Synthetic Tipping Point Data', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Physical Interpretation

### What does β tell us?

The steepness parameter β is related to the system's coupling strength:

$$
\beta \approx 2 \frac{J}{T}
$$

Where:
- **J**: Interaction/coupling strength
- **T**: Temperature (noise/randomness)

**Typical Ranges:**
- **β < 2**: Weakly coupled systems (smooth transitions)
- **β ≈ 4.2 (Φ³)**: Universal critical point
- **β > 6**: Strongly coupled systems (sharp transitions)
- **β > 10**: Extreme/catastrophic systems

In [ ]:
# Load real beta estimates
beta_estimates_path = Path('data/derived/beta_estimates.csv')

if beta_estimates_path.exists():
    df = pd.read_csv(beta_estimates_path)
    
    print("\n📊 Real-World β Estimates:")
    print("=" * 60)
    
    # Show examples from different domains
    examples = df.sample(min(10, len(df)), random_state=42)[['preset', 'beta', 'theta', 'domain']]
    print(examples.to_string(index=False))
    
    print("\n📈 Summary Statistics:")
    print(f"  β range: {df['beta'].min():.2f} → {df['beta'].max():.2f}")
    print(f"  β mean: {df['beta'].mean():.2f} ± {df['beta'].std():.2f}")
    print(f"  β median: {df['beta'].median():.2f}")
else:
    print("⚠ Beta estimates file not found. Using mock data.")
    # Create mock data
    df = pd.DataFrame({
        'preset': ['climate_amoc', 'llm_gpt', 'synapse_release'],
        'beta': [4.0, 4.2, 4.1],
        'theta': [0.18, 9.87, 0.48],
        'domain': ['Climate', 'AI/ML', 'Neuroscience']
    })
    print(df)

## 5. Exercise: Analyze Your Own System

Try fitting the UTAC model to your own data!

**Required:**
- Control parameter array `R` (e.g., temperature, resource availability)
- Response/order parameter array `sigma` (e.g., state variable, activation)

Uncomment and modify the code below:

In [ ]:
# # YOUR DATA HERE
# R_custom = np.array([...])  # Your control parameter
# sigma_custom = np.array([...])  # Your response variable
# 
# # Fit the model
# beta_custom, theta_custom, r2_custom = fit_sigmoid(R_custom, sigma_custom)
# 
# print(f"Your system: β = {beta_custom:.2f}, Θ = {theta_custom:.2f}")
# print(f"Quality of fit: R² = {r2_custom:.4f}")
# 
# # Interpret
# if beta_custom < 2:
#     print("→ Weakly coupled system (smooth transition)")
# elif beta_custom < 6:
#     print("→ Moderately coupled system (critical behavior)")
# else:
#     print("→ Strongly coupled system (sharp transition)")

## Summary

**What we learned:**
1. ✓ UTAC sigmoid function: σ(R; β, Θ) = 1/(1 + e^(-β(R-Θ)))
2. ✓ β controls transition steepness (coupling strength)
3. ✓ Θ marks the critical threshold
4. ✓ How to fit β and Θ to empirical data
5. ✓ Physical interpretation: β ≈ 2J/T

**Next Steps:**
- Tutorial 2: V6 Wavefunction and Ψ-Field Integration
- Tutorial 3: Genesis Cube and 4D Visualization
- Tutorial 4: Advanced Beta Extraction Techniques

---

**References:**
- Wei et al. 2022: "Emergent Abilities of Large Language Models"
- Global Tipping Points 2025
- Feldtheorie V6 Documentation: `docs/VISUALIZATION_INDEX.md`